<a href="https://colab.research.google.com/github/NguyenDoanBinhAn1714/Lab_2-AI_IOT-bailam/blob/main/lab4_AI_IOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Tạo cấu trúc thư mục chuẩn
dirs = ['data', 'src', 'models', 'outputs', 'figures']
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Đã tạo xong cấu trúc thư mục Lab 4.")

Đã tạo xong cấu trúc thư mục Lab 4.


In [2]:
%%writefile src/utils.py
import pandas as pd

def create_time_series_features(df):
    """Tạo các đặc trưng chuỗi thời gian cho Forecasting"""
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['lag_1'] = df['Appliances'].shift(1)
    df['rolling_mean_6'] = df['Appliances'].rolling(window=6).mean()
    df['delta_1'] = df['Appliances'].diff(1)

    # Target tương lai 1 bước (10 phút)
    df['target_future'] = df['Appliances'].shift(-1)

    return df.dropna()

def generate_risk_recommendation(predicted_value):
    """Chuyển đổi dự báo thành mức rủi ro và hành động"""
    if predicted_value > 250:
        return "CRITICAL", "TURN_OFF_NON_ESSENTIALS", "Dự báo vượt ngưỡng nguy hiểm"
    elif predicted_value > 120:
        return "WARNING", "MONITOR_AND_PREPARE", "Dự báo tiêu thụ tăng cao"
    else:
        return "NORMAL", "NO_ACTION", "Tiêu thụ ổn định"

Writing src/utils.py


In [3]:
%%writefile src/train_forecast.py
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from utils import create_time_series_features, generate_risk_recommendation

print("1. Đang tải và tiền xử lý dữ liệu...")
url = "https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv"
df = pd.read_csv(url)
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()

# Lấy 2000 dòng để demo chạy nhanh
df = df.iloc[-2000:].copy()
df = create_time_series_features(df)

print("2. Chia Train/Test theo thời gian...")
train_size = int(len(df) * 0.75)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

drop_cols = ['Appliances', 'target_future', 'rv1', 'rv2']
features = [col for col in df.columns if col not in drop_cols]

X_train, y_train = train_df[features], train_df['target_future']
X_test, y_test = test_df[features], test_df['target_future']

print("3. Train mô hình Linear Regression...")
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Đóng gói Model
bundle = {
    'model': model,
    'features': features,
    'version': 'linear_regression_v1'
}
joblib.dump(bundle, 'models/forecast_model_bundle_v1.joblib')

print("4. Đánh giá và lưu Metrics...")
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
bias = np.mean(y_pred - y_test)

metrics = {
    "MAE": round(mae, 4),
    "RMSE": round(rmse, 4),
    "Bias": round(bias, 4)
}
with open('outputs/forecast_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=4)

print("5. Sinh Forecast Log & Predictions...")
log_df = test_df[['Appliances']].copy()
log_df['actual_value'] = y_test
log_df['predicted_value'] = np.round(y_pred, 2)
log_df['forecast_error'] = log_df['predicted_value'] - log_df['actual_value']

decisions = log_df['predicted_value'].apply(generate_risk_recommendation)
log_df['risk_level'] = [d[0] for d in decisions]
log_df['recommendation'] = [d[1] for d in decisions]
log_df['reason'] = [d[2] for d in decisions]

log_df.to_csv('outputs/forecast_test_predictions.csv')
log_df.to_csv('outputs/forecast_log.csv')
print("✅ HOÀN TẤT TRAIN & LƯU FILE!")

Writing src/train_forecast.py


In [4]:
%%writefile src/app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
from typing import Dict, Any
from src.utils import generate_risk_recommendation

app = FastAPI(title="Forecasting API", description="API Dự báo tiêu thụ năng lượng")

# Tải mô hình
bundle = joblib.load('models/forecast_model_bundle_v1.joblib')
model = bundle['model']
features = bundle['features']

@app.get("/health")
def health_check():
    return {"model_loaded": True, "status": "running", "version": bundle['version']}

@app.post("/forecast")
def predict_forecast(data: Dict[str, float]):
    # Chuyển đổi JSON sang DataFrame
    input_df = pd.DataFrame([data])

    # Đảm bảo đủ cột features
    missing_cols = [col for col in features if col not in input_df.columns]
    if missing_cols:
        return {"error": f"Thiếu các cột: {missing_cols}"}

    # Dự báo
    pred_val = model.predict(input_df[features])[0]

    # Quyết định
    risk, rec, reason = generate_risk_recommendation(pred_val)

    return {
        "model_output": {
            "target": "Appliances",
            "forecast_horizon_minutes": 10,
            "predicted_value": round(pred_val, 2),
            "unit": "Wh per 10-minute interval",
            "model_version": bundle['version']
        },
        "decision": {
            "risk_level": risk,
            "recommendation": rec,
            "reason": reason,
            "safety_note": "Forecast output is a recommendation signal, not an automatic actuator command."
        }
    }

Writing src/app.py


In [5]:
import subprocess
import time
import requests
import nest_asyncio
import uvicorn
import threading
import json

# 1. Chạy quá trình Train
print("BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN")
!python src/train_forecast.py
print("-" * 50)

# 2. Khởi động API Server ngầm
nest_asyncio.apply()
from src.app import app

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="error")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

print("\nĐang khởi động FastAPI Server...")
time.sleep(3)

# 3. Test API bằng Request
print("\n--- TEST API /forecast ---")
# Giả lập payload lấy từ 1 mẫu thật trong dataset
payload = {
    "T1": 21.6, "RH_1": 45.3, "T2": 20.8, "RH_2": 42.5, "T3": 23.1, "RH_3": 42.1,
    "T4": 20.9, "RH_4": 42.1, "T5": 19.8, "RH_5": 50.1, "T6": 8.6, "RH_6": 48.6,
    "T7": 20.2, "RH_7": 36.2, "T8": 22.7, "RH_8": 44.5, "T9": 19.8, "RH_9": 41.5,
    "T_out": 8.6, "Press_mm_hg": 756.2, "RH_out": 84.1, "Windspeed": 4.0,
    "Visibility": 40.0, "Tdewpoint": 6.0, "hour": 15, "dayofweek": 3,
    "lag_1": 150.0, "rolling_mean_6": 120.0, "delta_1": 30.0
}

response = requests.post("http://127.0.0.1:8000/forecast", json=payload)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

# Lưu kết quả test API
with open("outputs/api_test_result.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, indent=4, ensure_ascii=False)

# Tắt server
server.should_exit = True
server_thread.join(timeout=2)

BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN
1. Đang tải và tiền xử lý dữ liệu...
2. Chia Train/Test theo thời gian...
3. Train mô hình Linear Regression...
4. Đánh giá và lưu Metrics...
5. Sinh Forecast Log & Predictions...
✅ HOÀN TẤT TRAIN & LƯU FILE!
--------------------------------------------------

Đang khởi động FastAPI Server...

--- TEST API /forecast ---
{
  "error": "Thiếu các cột: ['lights']"
}
